In [1]:
import os
import wandb
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hvasisht16 (hvasisht16-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# Get the dataset from UCI
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data -qq

# modified from https://github.com/dmlc/xgboost/blob/master/demo/multiclass_classification/train.py
# Import wandb
import wandb
import numpy as np
import xgboost as xgb

run = wandb.init(project="Lab1-visualize-models", name="xgboost")

# label need to be 0 to num_class -1
data = np.loadtxt('./dermatology.data', delimiter=',',
        converters={33: lambda x:int(x == '?'), 34: lambda x:int(x) - 1})
sz = data.shape

train = data[:int(sz[0] * 0.7), :]
test = data[int(sz[0] * 0.7):, :]

train_X = train[:, :33]
train_Y = train[:, 34]

test_X = test[:, :33]
test_Y = test[:, 34]

xg_train = xgb.DMatrix(train_X, label=train_Y)
xg_test = xgb.DMatrix(test_X, label=test_Y)
# setup parameters for xgboost
param = {}
# use softmax multi-class classification
param['objective'] = 'multi:softmax'
# scale weight of positive examples
param['eta'] = 0.1
param['max_depth'] = 6
param['silent'] = 1
param['nthread'] = 4
param['num_class'] = 6
wandb.config.update(param)

watchlist = [(xg_train, 'train'), (xg_test, 'test')]
num_round = 5

# Add the wandb xgboost callback
bst = xgb.train(param, xg_train, num_round, watchlist, callbacks=[wandb.xgboost.WandbCallback()])
# get prediction
pred = bst.predict(xg_test)
error_rate = np.sum(pred != test_Y) / test_Y.shape[0]
print('Test error using softmax = {}'.format(error_rate))

run.summary['Error Rate'] = error_rate

wandb.sklearn.plot_confusion_matrix(test_Y, pred, [0., 1., 2., 3., 4., 5.])

run.finish()

[0]	train-mlogloss:1.42585	test-mlogloss:1.57777
[1]	train-mlogloss:1.24766	test-mlogloss:1.39396
[2]	train-mlogloss:1.10264	test-mlogloss:1.24738
[3]	train-mlogloss:0.98193	test-mlogloss:1.12696
[4]	train-mlogloss:0.87951	test-mlogloss:1.03075


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:748: FutureWarning: Pass `evals` as keyword args.
  warnings.warn(msg, FutureWarning)
/usr/local/lib/python3.12/dist-packages/wandb/integration/xgboost/xgboost.py:126: UserWarning: [18:39:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "silent" } are not used.

  config = model.save_config()


Test error using softmax = 0.15454545454545454


epoch,▁▃▅▆█
test-mlogloss,█▆▄▂▁
train-mlogloss,█▆▄▂▁
Error Rate,0.15455
epoch,4
